# Etapa final — preflight antes do teste interno

Este notebook **não executa inferência e não abre imagens ou máscaras do teste**.

A verificação confere os hashes do protocolo, do manifesto, das configurações, dos checkpoints e dos pesos do SAM. O lacre produzido é exigido pelo notebook de avaliação interna.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
from pathlib import Path
import json, os, subprocess, sys

PROJECT_ROOT = Path('/content/drive/MyDrive/Projeto_TCC_Lucas')
PROTOCOL = PROJECT_ROOT / 'configs/final_internal_test_protocol_v1.yaml'
OUTPUT = PROJECT_ROOT / 'results/reproduction/internal_dfuc2022_preflight.json'
%cd {PROJECT_ROOT}
!pip -q install pyyaml pytest
print('Projeto:', PROJECT_ROOT)
print('Protocolo:', PROTOCOL)
print('Nenhuma imagem de teste será aberta neste notebook.')


## 1. Conferir as regras congeladas

Os testes abaixo verificam a matriz de dez saídas, a seed 42, os cinco contrastes primários, a separação da DRIFTB exploratória e a proibição de escolher algo pelo teste.


In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_ROOT / "src") + os.pathsep + env.get("PYTHONPATH", "")
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_final_internal_test_protocol.py"],
    text=True,
    capture_output=True,
    env=env,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()


## 2. Selar os arquivos

O programa calcula hashes dos arquivos. Para o manifesto de teste, somente os bytes do próprio CSV são usados para conferir integridade; suas linhas não são lidas e nenhuma imagem ou máscara é carregada.


In [ ]:
command = [
    sys.executable,
    "scripts/audit_final_internal_test_protocol.py",
    "--project-root",
    str(PROJECT_ROOT),
    "--protocol",
    str(PROTOCOL),
    "--output",
    str(OUTPUT),
]
result = subprocess.run(command, text=True, capture_output=True, env=env)
print(result.stdout)
print(result.stderr)
result.check_returncode()


## 3. Resumo legível

A verificação deve indicar `passed`, dez braços, `scientific_data_accessed = false` e `inference_executed = false`. Se faltar um checkpoint, confira o caminho da execução registrado no protocolo.


In [ ]:
report = json.loads(OUTPUT.read_text(encoding="utf-8"))
print("Status:", report["status"])
print("Dados científicos acessados:", report["scientific_data_accessed"])
print("Inferência executada:", report["inference_executed"])
print("Hash do protocolo:", report["protocol_sha256"])
print("Hash do manifesto:", report["test_manifest"]["sha256"])
print("\nArquivos selados:")
for arm in report["arms"]:
    key = "checkpoint_sha256" if "checkpoint_sha256" in arm else "sam_checkpoint_sha256"
    print(f"- {arm['id']}: {arm[key][:16]}…")


## 4. Arquivo de verificação preliminar

O arquivo JSON registra os hashes conferidos. A avaliação interna usa o mesmo protocolo e exige esses hashes.


In [ ]:
from google.colab import files

files.download(str(OUTPUT))
